# 05 · Verificación del masking de dni y salario
Córrelo dos veces: con un usuario de `electrocasa_ingenieria` (ve valores reales) y con uno que no lo sea
(ve `*****123` y `NULL`). Si solo tienes un usuario, sácate temporalmente de `electrocasa_ingenieria`,
espera ~1 minuto y vuelve a correrlo. Captura ambos resultados para el README.

In [ ]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo del proyecto")
catalogo = dbutils.widgets.get("catalog")
print("Catálogo:", catalogo)

In [ ]:
sql = """
SELECT current_user() AS usuario,
       is_account_group_member('electrocasa_ingenieria') AS es_ingenieria;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Tabla de Silver con MASK
SELECT id_empleado, nombre, dni, salario, sucursal_id, valido_desde, valido_hasta
FROM ${electrocasa.catalog}.silver.empleados
ORDER BY id_empleado, version LIMIT 10;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Tabla DERIVADA en Gold: conserva los valores reales para ingeniería porque el dueño del
-- pipeline es miembro de electrocasa_ingenieria; para el resto, la MASK de gold los oculta.
SELECT id_empleado, nombre, dni, salario, nombre_sucursal
FROM ${electrocasa.catalog}.gold.empleados_vigentes
ORDER BY id_empleado LIMIT 10;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)

In [ ]:
sql = """
-- Las máscaras declaradas en cada columna
DESCRIBE TABLE EXTENDED ${electrocasa.catalog}.gold.empleados_vigentes;
""".replace("${electrocasa.catalog}", catalogo)
resultado = None
for sentencia in [s.strip() for s in sql.split(";")]:
    if sentencia and not all(l.strip().startswith("--") for l in sentencia.splitlines() if l.strip()):
        resultado = spark.sql(sentencia)
if resultado is not None and len(resultado.columns) > 0:
    display(resultado)